### Imports

In [1]:
import sys
import os

sys.path.append(os.path.abspath("..")) 

import json
import pandas as pd
from data_class.raw_data import RawData
from tqdm.auto import tqdm
from clean_text import clean_text
from format_date import format_date

### Helper functions for cleaning

In [2]:
def clean_article(article: RawData):
    """Clean a single article entry"""
    cleaned: RawData = article.copy()
    
    # Clean date
    cleaned['publish_date'] = format_date(cleaned['publish_date'])
    
    # Clean text fields
    for field in ['title', 'content', 'claim']:
        if field in cleaned and cleaned[field]:
            cleaned[field] = clean_text(cleaned[field])
    
    # Ensure authors is a list
    if 'authors' in cleaned and cleaned['authors'] is None:
        cleaned['authors'] = []

    cleaned["source_bias"] = "LEAST-BIASED"

    # Make other props upper case
    cleaned["source"] = cleaned["source"].upper()
    cleaned["type"] = cleaned["type"].upper()
    
    return cleaned

### Load in dataset

In [3]:
with open('../outputs/factcheckorg-factcheck.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

### Clean 

In [4]:
# Clean all articles
cleaned_data = [clean_article(article) for article in tqdm(data, desc="Cleaning articles")]

Cleaning articles:   0%|          | 0/4420 [00:00<?, ?it/s]

### Convert to DF and inspect

In [5]:
df = pd.DataFrame(cleaned_data)

print(f"\nSample of cleaned data:")
print(df[['title', 'content', 'publish_date', 'source_bias']].head())


Sample of cleaned data:
                                               title  \
0            Sorting out the Facts on Epstein Claims   
1  Experts Say Democratic Video Not ‘Seditious,’ ...   
2  Trump Misrepresents Biden’s Job Numbers, SNAP ...   
3                      Our Annual Fundraising Appeal   
4  Revised CDC Website About Autism and Vaccines ...   

                                             content  \
0  The House voted nearly unanimously on Nov. 18 ...   
1  After six congressional Democrats released a v...   
2  Addressing a meeting of McDonald’s restaurant ...   
3  Today, we’re launching our annual end-of-year ...   
4  Under Robert F. Kennedy Jr., a longtime anti-v...   

                publish_date   source_bias  
0  2025-11-25T22:46:15+00:00  LEAST-BIASED  
1  2025-11-24T23:07:48+00:00  LEAST-BIASED  
2  2025-11-21T22:36:37+00:00  LEAST-BIASED  
3  2025-11-21T21:05:09+00:00  LEAST-BIASED  
4  2025-11-20T23:44:16+00:00  LEAST-BIASED  


### Save cleaned data to separate JSON files

In [6]:
output_dir = '../outputs_clean/factcheckorg'

# Create the output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

with open(f'{output_dir}/factcheckorg.json', 'w', encoding='utf-8') as f:
    json.dump(cleaned_data, f, indent=2, ensure_ascii=False)

print("Cleaning complete! Files saved.")

Cleaning complete! Files saved.
